# Smooth segmentation (experiment)

Boxes are cheap and good enough to *start*. This notebook explores the next
step: **smooth object masks** so the scan highlight hugs the object instead of
being a block.

Plan: keep boxes as the default, and offer smooth masks as a **scan action**
(prompted, on one object) where we can afford it.

| Model | Prompt | CPU latency* | Notes |
|---|---|---|---|
| YOLO11n | none | ~50 ms (OpenVINO) | boxes only |
| FastSAM-s | none | ~0.5-2 s | all masks, no class |
| MobileSAM | point | ~1-2 s | fast promptable |
| **SAM 2.1 tiny** | point/box | ~2 s | good boundaries |
| SAM ViT-B | point/box | ~10 s | best, too slow on CPU |

\* measured on this machine, CPU, as a rough guide. The GPU step at the end is
where these get usable.

Everything below starts from a YOLO box, then asks a SAM model to turn that box
into a smooth mask.

## 0. Setup

In [ ]:
# %pip install -r requirements.txt

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import torch
from ultralytics import YOLO

%matplotlib inline

im = Image.open("samples/bus.jpg").convert("RGB")
img = np.array(im)
H, W = img.shape[:2]
print("image", im.size)

det = YOLO("yolo11n.pt")

## 1. Start from a detection box

In [ ]:
res = det.predict(img, imgsz=640, verbose=False)[0]

TARGET = "bus"
box = None
for b in res.boxes:
    if res.names[int(b.cls)] == TARGET:
        box = b.xyxy[0].tolist()
        break
assert box is not None, f"no '{TARGET}' found"
box = [float(v) for v in box]
cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
print("box", [round(v) for v in box], "centre", (round(cx), round(cy)))

plt.figure(figsize=(9, 6))
plt.imshow(res.plot()[:, :, ::-1])
plt.axis("off")
plt.title(f"YOLO box for '{TARGET}'")

## 2. Helpers

`get_masks` normalises whatever a model returns into boolean arrays at the
image size. `outline` draws the smooth contour - this is what the HUD would
render.

In [ ]:
GREEN = (0, 255, 170)


def get_masks(result, image_hw):
    '''Return a list of boolean HxW masks from an Ultralytics result.'''
    if result.masks is None:
        return []
    data = result.masks.data.detach().cpu().numpy()
    if data.ndim == 2:
        data = data[None]
    out = []
    for m in data:
        if m.shape != image_hw:
            m = cv2.resize(m, (image_hw[1], image_hw[0]), interpolation=cv2.INTER_LINEAR)
        out.append(m > 0.5)
    return out


def iou(a, b):
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return inter / union if union else 0.0


def box_mask(box, shape):
    m = np.zeros(shape, bool)
    x0, y0, x1, y1 = (int(v) for v in box)
    m[max(y0, 0):y1, max(x0, 0):x1] = True
    return m


def outline(image, mask, colour=GREEN, thickness=3, fill_alpha=0.28):
    '''Smooth coloured outline + translucent fill, like the scan visor.'''
    out = image.copy()
    m = mask.astype(np.uint8)
    if fill_alpha:
        out[m > 0] = (out[m > 0] * (1 - fill_alpha) + np.array(colour) * fill_alpha).astype(np.uint8)
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, contours, -1, colour, thickness, lineType=cv2.LINE_AA)
    return out

## 3. FastSAM-s (class-agnostic, no prompt)

In [ ]:
from ultralytics import FastSAM

fastsam = FastSAM("FastSAM-s.pt")
t0 = time.time()
fs = fastsam(img, imgsz=1024, conf=0.35, iou=0.9, verbose=False)[0]
print(f"FastSAM: {len(fs.boxes)} masks in {time.time()-t0:.2f}s")

m = box_mask(box, (H, W))
fs_masks = get_masks(fs, (H, W))
centre_masks = [k for k in fs_masks if k[int(cy), int(cx)]]
if centre_masks:
    fs_best = max(centre_masks, key=lambda k: k.sum())
elif fs_masks:
    fs_best = max(fs_masks, key=lambda k: iou(k, m))
else:
    fs_best = None
if fs_best is not None:
    print("fastsam mask area:", int(fs_best.sum()), "px",
      "(class-agnostic; it does not necessarily isolate the whole bus)")
    plt.figure(figsize=(9, 6))
    plt.imshow(outline(img, fs_best))
    plt.axis("off")
    plt.title("FastSAM-s mask for the bus region")

## 4. MobileSAM (point prompt - the reticle case)

In [ ]:
from ultralytics import SAM

mobile = SAM("mobile_sam.pt")
t0 = time.time()
ms = mobile(img, points=[[int(cx), int(cy)]], labels=[1], verbose=False)[0]
print(f"MobileSAM: {time.time()-t0:.2f}s")
ms_masks = get_masks(ms, (H, W))
plt.figure(figsize=(9, 6))
plt.imshow(outline(img, ms_masks[0], colour=(120, 200, 255)))
plt.axis("off")
plt.title("MobileSAM point prompt")

## 5. SAM 2.1 tiny (box prompt - sharper boundaries)

In [ ]:
sam2 = SAM("sam2.1_t.pt")
t0 = time.time()
s2 = sam2(img, bboxes=[box], verbose=False)[0]
print(f"SAM 2.1 tiny: {time.time()-t0:.2f}s")
s2_masks = get_masks(s2, (H, W))
plt.figure(figsize=(9, 6))
plt.imshow(outline(img, s2_masks[0], colour=(255, 180, 80)))
plt.axis("off")
plt.title("SAM 2.1 tiny box prompt")

## 6. Overlay comparison

In [ ]:
def contour_mask(mask, colour):
    out = np.zeros((H, W, 3), np.uint8)
    ct, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, ct, -1, colour, 3, lineType=cv2.LINE_AA)
    return out


stack = img.copy()
if fs_best is not None:
    stack = cv2.addWeighted(stack, 1.0, contour_mask(fs_best, (0, 255, 170)), 0.9, 0)
if ms_masks:
    stack = cv2.addWeighted(stack, 1.0, contour_mask(ms_masks[0], (120, 200, 255)), 0.9, 0)
if s2_masks:
    stack = cv2.addWeighted(stack, 1.0, contour_mask(s2_masks[0], (255, 180, 80)), 0.9, 0)

plt.figure(figsize=(11, 7))
plt.imshow(stack)
plt.axis("off")
plt.title("green=FastSAM   blue=MobileSAM   orange=SAM 2.1")

## 7. Smooth outline as the HUD highlight

This is the visual we actually want on the phone: a clean, closed, coloured
boundary that hugs the object. The same mask is what we would unproject for the
3D shell.

In [ ]:
plt.figure(figsize=(10, 7))
plt.imshow(outline(img, s2_masks[0], colour=GREEN, thickness=4, fill_alpha=0.2))
plt.axis("off")
plt.title("proposed scan highlight (SAM 2.1 mask)")

## 8. Mask-based 3D shell (vs the old box cloud)

Previously we unprojected the whole **box**, which includes background and made
a blob. Using the **mask** keeps only the object's pixels.

Two honest caveats:
- Depth Anything V2 is *relative* depth, so the shell's absolute scale is wrong
  until we anchor it (AR plane, known size, ...).
- A single view only ever gives you a shell "from one direction" - that is
  exactly what we want for an outline, but not a full 3D model.

In [ ]:
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

DEPTH_ID = "depth-anything/Depth-Anything-V2-Small-hf"
dproc = AutoImageProcessor.from_pretrained(DEPTH_ID)
dmodel = AutoModelForDepthEstimation.from_pretrained(DEPTH_ID).eval()


@torch.no_grad()
def depth_map(image):
    inputs = dproc(images=image, return_tensors="pt")
    out = dmodel(**inputs).predicted_depth
    out = torch.nn.functional.interpolate(
        out.unsqueeze(1), size=(image.height, image.width), mode="bicubic", align_corners=False
    )
    return out.squeeze().numpy()


def intrinsics_from_fov(w, h, fov_deg=60.0):
    f = 0.5 * w / np.tan(np.radians(fov_deg) / 2)
    return f, f, w / 2, h / 2


def cloud_from_mask(mask, depth, stride=3, scale=1.0):
    fx, fy, cx_, cy_ = intrinsics_from_fov(W, H)
    ys, xs = np.nonzero(mask[::stride, ::stride])
    ys = ys * stride
    xs = xs * stride
    z = depth[ys, xs] * scale
    x = (xs - cx_) * z / fx
    y = (ys - cy_) * z / fy
    rgb = img[ys, xs] / 255.0
    return x, y, z, rgb


depth = depth_map(im)
# normalise the relative depth into a nicer range for display
depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-6)
print("depth range", float(depth.min()), float(depth.max()))

# box cloud (old way) vs mask cloud (new way)
bx, by, bz, brgb = cloud_from_mask(box_mask(box, (H, W)), depth, stride=6, scale=2.0)
mx, my, mz, mrgb = cloud_from_mask(s2_masks[0], depth, stride=4, scale=2.0)
print("box points", len(bx), "| mask points", len(mx))

fig = plt.figure(figsize=(13, 6))
for i, (title, (x, y, z, c)) in enumerate(
    [("box cloud (old)", (bx, by, bz, brgb)), ("mask shell (new)", (mx, my, mz, mrgb))]
):
    ax = fig.add_subplot(1, 2, i + 1, projection="3d")
    ax.scatter(x, y, z, c=c, s=2, depthshade=False)
    ax.invert_yaxis()
    ax.view_init(elev=12, azim=-70)
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("depth")
    try:
        ax.set_box_aspect((1, 1.4, 0.5))
    except Exception:
        pass
plt.tight_layout()

## 9. OpenVINO on the iGPU

We confirmed the iGPU shows up and is ~2.3x faster than CPU for YOLO11n. Run the
standalone check for the full table:

```bash
../.venv/Scripts/python.exe ../scripts/openvino_check.py --runs 15
```

In the notebook you can list devices directly:

In [ ]:
import openvino as ov

core = ov.Core()
print("OpenVINO", ov.__version__)
print("devices:", core.available_devices)
for d in core.available_devices:
    try:
        print(f"  {d:<6} {core.get_property(d, 'FULL_DEVICE_NAME')}")
    except Exception:
        pass

## 10. Takeaways

- **Boxes stay the default** for speed.
- **SAM 2.1 tiny** looks like the sweet spot for a smooth scan highlight:
  decent boundaries, ~2 s on CPU, and it accepts a box prompt (so we can feed it
  a YOLO box) or a point prompt (reticle).
- **FastSAM** is best when we need to mask the whole frame with no prompt.
- Next: export SAM 2.1 / FastSAM to OpenVINO so the smooth option is actually
  affordable on the iGPU, then decide the `/scan` API (return boxes always, and
  masks on demand).